In this notebook, you will

- build on your existing knowledge of PyTorch and deep learning basics
- prior knowledge of reinforcement learning is helpful but not required


Focus on
- easy to understand, incremental steps vs. mathematical rigor
- simplify (without trivializing) the algorithms when possible
- narrow slice of reinforcement learning

This notebook is identical to `nanoRLHF.ipynb` except that instead of using a CVC regex to select liked names, the **human preference** signal comes from [Pokémon names](https://gist.githubusercontent.com/ralts00/31415709fb34c1b2ec556c396efc3d80/raw/516ef1179f10f4a0ecb4f50f118e6757fef85243/pokemon_names.txt). The goal is to fine-tune the character-level model so that it generates names that *sound like* Pokémon.

For a more mathematical treatment of reinforcement learning, policy gradients, and proximal policy optimization, check out:

- [Sutton & Barto, "Reinforcement Learning: An Introduction"](http://incompleteideas.net/book/the-book-2nd.html) — the classic RL textbook, especially Chapters 13 and 14 for policy gradient methods
- [OpenAI Spinning Up: Policy Gradient Methods](https://spinningup.openai.com/en/latest/spinningup/rl_intro.html#policy-gradient-methods) — practical and mathematical overview
- [Proximal Policy Optimization Algorithms (Schulman et al., 2017)](https://arxiv.org/abs/1707.06347) — original PPO paper
- [Lil'Log: Proximal Policy Optimization Explained](https://lilianweng.github.io/posts/2018-06-24-policy-gradient/#ppo) — blog post with clear math and diagrams

Before we get started, let's make sure you have the correct version of PyTorch installed. If you have an older version or want to start fresh, you can uninstall the current PyTorch, torchvision, and torchaudio packages using the following command.

In [ ]:
# !pip3 uninstall -y torch torchvision torchaudio

If you need to install PyTorch with CUDA support (for GPU acceleration), you can use the following command. Make sure your environment supports CUDA 11.8.

In [ ]:
# !pip3 install torch --index-url https://download.pytorch.org/whl/cu118

Let's import PyTorch and check the version to make sure everything is set up correctly.

In [ ]:
import torch as pt
pt.__version__

Now, let's set up the device for computation. We'll use the GPU if it's available, otherwise we'll fall back to the CPU. The `nvidia-smi` command will show you information about your GPU if one is present.

In [ ]:
device = "cuda" if pt.cuda.is_available() else "cpu"
!nvidia-smi
device

We'll use a dataset of names for our experiments. Let's fetch a list of names from GitHub using Python's `urllib` library. This will give us a simple, real-world dataset to work with.

In [ ]:
import urllib
with urllib.request.urlopen('https://raw.githubusercontent.com/karpathy/makemore/master/names.txt') as resp:
  src = resp.read().decode('utf-8')
src

Now, let's convert the downloaded text into a Python list of names. We'll take a quick look at the first few names to make sure everything looks good.

In [ ]:
names = src.splitlines()
names[:5]

Next, we'll extract the unique characters (tokens) used in the dataset. We'll also add special tokens for the start (`_`) and end (`.`) of a name. This will help us later when we encode and decode names.

In [ ]:
tokens = "_" + "." + "".join(sorted(set("".join(names))))
tokens

To work with our tokens, we'll create two dictionaries: `stoi` (string-to-index) and `itos` (index-to-string). These will let us easily convert between characters and their corresponding indices.

In [ ]:
stoi = {v:k for k,v in enumerate(tokens)}

In [ ]:
print(stoi)

In [ ]:
itos = {k:v for k,v in enumerate(tokens)}

In [ ]:
print(itos)

Let's define two simple functions: `enc` to encode a name as a list of indices, and `dec` to decode a list of indices back to a string. This will make it easy to switch between string and numeric representations.

In [ ]:
enc = lambda name: [stoi[s] for s in name]
dec = lambda chars: "".join(itos[i] for i in chars)

In [ ]:
dec(enc('_emma.'))

Now, let's create a function `stot` that converts a string into a one-hot encoded tensor. This will be useful for representing names in a format suitable for neural networks.

In [ ]:
stot = lambda name: pt.nn.functional.one_hot(pt.tensor(enc(name)), len(tokens))

In [ ]:
stot('_emma.')

To go back from a one-hot encoded tensor to a string, we'll define the `ttos` function. This will help us interpret the outputs of our model later.

In [ ]:
ttos = lambda x: dec(x.argmax(-1).tolist())

In [ ]:
ttos(stot('_emma.'))

Define a constant `CTX_SZ` to represent a window size used by the model to predict the next character. For example, in a name `olivia`, the window
- `oliv` predicts the token `i`
- `livi` predicts the token `a`
- `CTX_SZ` start of name tokens, i.e. `____` predict the token `o`

In [ ]:
CTX_SZ = 4
CTX_SZ

With `CTX_SZ` constant defined, you can illustrate how a name like `emma` is represented using the sparse embedding with `stot`.

Create a tensor called `name` using the `CTX_SZ` start tokens `_` and a single end token`.` Also, report on the shape of the `name` tensor

In [ ]:
name = stot("".join(CTX_SZ * "_" + "emma" + "."))
name, name.shape

The rest of the notebook will depart from common machine learning terminology to make the upcoming use of reinforcement learning easier to understand. The notebook will use the term **observations** to describe the inputs to the machine learning model (aka `X` or `obs`) while the outputs of the model will be described as **actions** (aka `y`). You are probably comfortable with the idea of training your machine learning model on `(X,y)` pairs, so the following will use the `(obs, action)` pairs. You can think of the action as the token that the model predicts as its output. Hence, generating the name is like making a series of actions to pick the right tokens.

PyTorch has a convenient `unfold` function to take a tensor like `name` and convert it to a sequence of observations, each with a window length of `CTX_SZ`. For example, you can `unfold` the `name` tensor along the first (0th) dimension using the step size of 1. For convenience, permute the shape of the resulting tensor to swap the last two dimensions.

In [ ]:
name.unfold(0, CTX_SZ, 1).permute(0, 2, 1)

To make the output of the `unfold` clearer you can use `ttos` to change the tensors back to string. For a name like `emma` you should get observations like

- `___e`
- `__em`
- `_emm`
- etc

**NOTE:** Don't forget to drop the last token `.`

In [ ]:
[ttos(x) for x in name.unfold(0, CTX_SZ, 1)[:-1, :, :].permute(0, 2, 1)]

Since you are going to need to `unfold` all of the names in the data set, create a function `name_to_obs` that implements this capability.

In [ ]:
name_to_obs = lambda name: stot(name).unfold(0, CTX_SZ, 1)[:-1, :, :].permute(0, 2, 1)

In [ ]:
name_to_obs('____emma.').shape

Next, let's create another function `name_to_action` that converts a name to all the action (i.e. output tokens) that should be predicted by the model.

In [ ]:
name_to_action = lambda name: stot(name)[CTX_SZ:]

In [ ]:
y = name_to_action('____emma.')
y.shape

In [ ]:
ttos(name_to_action('____emma.'))

Now you are ready to create the data set to train your model.

- convert all the names in `names` to the observations
- use `cat` to concatenate all the sparse embeddings of the names to a data set tensor `X_data`

In [ ]:
X_data = [name_to_obs(CTX_SZ * "_" + name + ".") for name in names]
X_data = pt.cat(X_data)

In [ ]:
[ttos(x) for x in X_data[:10]], X_data.shape

Move the `X_data` tensor to your `device`

In [ ]:
X_data = X_data.to(device)

Next, create the `y_data` tensor using `name_to_action`

In [ ]:
y_data = pt.cat([name_to_action(CTX_SZ * "_" + name + ".") for name in names])

In [ ]:
ttos(y_data[:10]), y_data.shape

Move the `y_data` tensor to your device

In [ ]:
y_data = y_data.to(device)

The data set should be shuffled before use for training. Use the `randperm` function to shuffle both the `X_data` and the `y_data` tensors along the 0th dimension.

**NOTE:** Don't forget to set the seed using `manual_seed`.

In [ ]:
pt.manual_seed(42)
idx = pt.randperm(len(y_data))
idx.shape

In [ ]:
X_data, y_data = X_data[idx], y_data[idx]
X_data.shape, y_data.shape

Since the data set is fairly large, let's use a 90%, 5%, 5% split for the training, validation, and test data sets respectively.

In [ ]:
val_idx, test_idx = int(len(X_data) * .9), int(len(X_data) * .95)
X_train, y_train = X_data[:val_idx], y_data[:val_idx]
X_val, y_val = X_data[val_idx:test_idx], y_data[val_idx:test_idx]
X_test, y_test = X_data[test_idx:], y_data[test_idx:]

X_train.shape, X_val.shape, X_test.shape

In [ ]:
class CharModel(pt.nn.Module):
  def __init__(self, tokens_sz, ctx_sz, emb_sz, head_sz, n_heads, device):
    super().__init__()

    self.tok_emb = pt.nn.Embedding(tokens_sz, emb_sz, device = device)
    self.pos_emb = pt.nn.Embedding(ctx_sz, emb_sz, device = device)
    self.pos_idx = pt.arange(ctx_sz, device = device)

    self.kw = pt.nn.Linear(emb_sz, head_sz, device = device, bias = False)
    self.qw = pt.nn.Linear(emb_sz, head_sz, device = device, bias = False)
    self.vw = pt.nn.Linear(emb_sz, head_sz, device = device, bias = False)
    self.mhsa = pt.nn.MultiheadAttention(head_sz, n_heads, batch_first = True, device = device)
    self.mhsa_ln = pt.nn.LayerNorm(head_sz, device = device)
    self.flatten = pt.nn.Flatten(1)
    self.relu = pt.nn.ReLU()
    self.mhsa_head = pt.nn.Linear(head_sz * ctx_sz, tokens_sz, device = device)


  def forward(self, x):
    x = self.tok_emb(x) + self.pos_emb(self.pos_idx)
    q, k, v = self.qw(x), self.kw(x), self.vw(x)
    x, _ = self.mhsa(q, k, v,  )
    x = self.mhsa_ln(x)
    x = self.flatten(x)
    x = self.relu(x)
    x = self.mhsa_head(x)
    return x


In [ ]:
EMB_SZ = 32
MHSA_HEAD_SZ = 64
N_HEADS = 2
nn = CharModel(len(tokens), CTX_SZ, EMB_SZ, MHSA_HEAD_SZ, N_HEADS, device)
nn

In [ ]:
ctx = pt.stack([stot("_" * CTX_SZ).argmax(-1).to(device)])
ctx

In [ ]:
nn(ctx).shape

In [ ]:
X_batch, y_batch = X_train.argmax(-1), y_train.argmax(-1)
import copy
model = copy.deepcopy(nn)
optim = pt.optim.AdamW(model.parameters())
optim

In [ ]:
def generate(model, seed = 42, device = device, ctx_sz = 4, num_samples = 10, max_tokens = 16):
  pt.manual_seed(seed)
  samples = []
  ctx = pt.stack([stot(ctx_sz * "_").argmax(-1).to(device)])
  for _ in range(num_samples):
    sample = ctx.clone()
    for _ in range(max_tokens):
      logits = model(sample[:, -ctx_sz:])
      probs = pt.nn.functional.softmax(logits, -1)
      tok = pt.multinomial(probs, 1)
      if tok.item() == 1:
        break
      sample = pt.cat([sample, tok], 1)

    sample = dec(sample.squeeze().tolist()[ctx_sz:])
    samples.append(sample)
  return samples

In [ ]:
EPOCHS = 2000
for epoch in range(EPOCHS):
  logits = model(X_batch)
  loss = pt.nn.functional.cross_entropy(logits, y_batch)
  if epoch % 100 == 0:
    with pt.no_grad():
      val_loss = pt.nn.functional.cross_entropy(model(X_val.argmax(-1)), y_val.argmax(-1))
      print(f"epoch {epoch:2d} train loss {loss.item():.4f} val loss {val_loss.item():.4f}")
      samples = generate(model, seed = epoch, device = device)
      print(",".join(samples))
      print()
  loss.backward()
  optim.step()
  optim.zero_grad()

# Human preferences for reinforcement learning with human feedback

Instead of using a CVC regex to select liked names from the training dataset, we fetch **Pokémon names** as the human preference signal. The idea is to fine-tune the model so that it generates names that *sound like* Pokémon.

We lowercase all Pokémon names and filter out any that contain characters outside `a-z` (e.g. hyphens, periods, spaces) so they are compatible with our token vocabulary.

In [ ]:
import re
import urllib

with urllib.request.urlopen('https://gist.githubusercontent.com/ralts00/31415709fb34c1b2ec556c396efc3d80/raw/516ef1179f10f4a0ecb4f50f118e6757fef85243/pokemon_names.txt') as resp:
  pokemon_src = resp.read().decode('utf-8')

# lowercase all names and keep only those with a-z characters
liked_names = [name.lower() for name in pokemon_src.splitlines() if name.strip()]
liked_names = [name for name in liked_names if bool(re.match('^[a-z]+$', name))]
print(f"{len(liked_names)} Pokémon names after filtering")
liked_names[:10]

In [ ]:
names = [CTX_SZ * "_" + name + "." for name in liked_names]
names[:5]

In [ ]:
obs = pt.cat([name_to_obs(name).argmax(-1) for name in names])
obs = obs.to(device)
obs.shape

In [ ]:
actions = pt.cat([name_to_action(name).argmax(-1) for name in names])
actions = actions.to(device)
actions.shape

In [ ]:
def reward(name, gamma = .9):
  idx = pt.arange(len(name) - 1, -1, -1).to(device)
  return gamma ** idx

reward(names[0][CTX_SZ:])

In [ ]:
rewards = pt.cat([reward(name[CTX_SZ:]) for name in names])
rewards[:10], actions[:10]

In [ ]:
import copy
rl_model = copy.deepcopy(model)
rl_model.train(False)
rl_optim = pt.optim.AdamW(rl_model.parameters())
rl_optim.zero_grad()

### Basic Policy Gradient Reinforcement Learning
* aka REINFORCE or vanilla policy gradient

In [ ]:
STEPS = 25
for step in range(STEPS):
  logits = rl_model(obs)
  log_prob_dist = pt.nn.functional.log_softmax(logits, -1)
  log_probs = log_prob_dist[pt.arange(len(actions)), actions]

  #policy gradient
  loss = -(rewards * log_probs).mean()

  with pt.no_grad():
    names = generate(rl_model, device = device, seed = 42 + step)
    new_names = set(set(names) - set(liked_names))
    print(f"step = {step:2d} loss={loss.item():.4f} net_new_names={len(new_names):2d}")
    print(",".join(new_names))

  loss.backward()
  rl_optim.step()
  rl_optim.zero_grad()

### Proximal Policy Optimization (PPO) Reinforcement Learning

In [ ]:
rl_model = copy.deepcopy(model)
rl_model.train(False)

ref_model = copy.deepcopy(model)
rl_optim = pt.optim.Adam(rl_model.parameters())
rl_optim.zero_grad()

In [ ]:
STEPS = 25
for step in range(STEPS):
  logits = rl_model(obs)
  log_probs = pt.nn.functional.log_softmax(logits, -1)[pt.arange(len(actions)), actions]
  with pt.no_grad():
    ref_log_probs = pt.nn.functional.log_softmax(ref_model(obs), -1)[pt.arange(len(actions)), actions]

  ratio = log_probs - ref_log_probs
  ratio = ratio.exp()

  ppo_loss1 = rewards * ratio
  ppo_loss2 = rewards * pt.clamp(ratio, .8, 1.2)

  loss = -pt.min(ppo_loss1, ppo_loss2).mean()

  log_lik_loss = -(rewards * log_probs).mean()

  with pt.no_grad():
    names = generate(rl_model, device = device, seed = 42 + step)
    new_names = set(set(names) - set(liked_names))
    print(f"step = {step:2d} ppo loss={loss.item():.4f} log likelihood {log_lik_loss:.4f} net_new_names={len(new_names):2d}")
    print(",".join(new_names))

  loss.backward()
  rl_optim.step()
  rl_optim.zero_grad()

## Evolution Strategies (ES): gradient-free fine-tuning

**What this adds (in 2 cells):**
- **No backprop**: we never call `.backward()` or use an optimizer.
- We perturb model **weights** with small Gaussian noise, evaluate a **scalar preference reward**, restore, and then **aggregate** the best directions.
- **Deterministic (greedy) decoding** for evaluation → exploration is only in **parameter space** (not per-token).
- Uses the **same Pokémon name preference** signal defined earlier for RL.

In [ ]:
# === Evolution Strategies (ES) helpers (matches the earlier reward usage) ===
import copy, numpy as np
import torch as pt

# ES objective: SAME as earlier RL parts — dataset log-likelihood weighted by `rewards`
# Uses the precomputed tensors: obs (B, T_ctx), actions (B,), rewards (B,)
def es_objective(model) -> float:
    model.eval()
    with pt.no_grad():
        logits = model(obs)  # (B, vocab)
        log_probs = pt.nn.functional.log_softmax(logits, dim=-1)[pt.arange(len(actions)), actions]
        score = (rewards * log_probs).mean().item()
    return float(score)

# Iterate parameters in a stable order
def _iter_params(m):
    for n, p in m.named_parameters():
        if p.requires_grad:
            yield n, p

# Safe in-place perturb/restore (no autograd on weight edits)
def _perturb_inplace(model, sigma):
    noises = []
    with pt.no_grad():
        for n, p in _iter_params(model):
            eps = pt.randn(p.shape, device=p.device)
            p.add_(sigma * eps)
            noises.append((n, eps))
    return noises

def _restore_inplace(model, noises, sigma):
    with pt.no_grad():
        for (n, eps), (_, p) in zip(noises, _iter_params(model)):
            p.sub_(sigma * eps)

# Aggregate weighted update by replaying the same RNG stream (via global seeding)
def _aggregate_update(model, seeds, z, sigma, lr):
    alpha = lr / len(seeds)
    with pt.no_grad():
        for seed, zi in zip(seeds, z):
            pt.manual_seed(int(seed))      # deterministic replay
            w = alpha * float(zi)
            for n, p in _iter_params(model):
                eps = pt.randn(p.shape, device=p.device)
                p.add_(w * eps)

In [ ]:
# === Simple ES training loop (didactic, objective-aligned) ===
pt.manual_seed(1337)

es_model = copy.deepcopy(model)   # start from your supervised model checkpoint
es_model.train(False)

POP        = 30       # population size
ITERS      = 120       # iterations
SIGMA      = 1e-2     # noise scale
LR         = 5e-3     # step size
EVAL_EVERY = 5

before = es_objective(es_model)
print(f"Before ES objective (reward-weighted log-lik): {before:.4f}")

for t in range(1, ITERS + 1):
    seeds = pt.randint(0, 2**31 - 1, (POP,))
    fit_scores = []

    # 1) Evaluate perturbed individuals on the SAME dataset objective
    for i in range(POP):
        pt.manual_seed(int(seeds[i]))     # deterministic perturbations
        noises = _perturb_inplace(es_model, SIGMA)

        s = es_objective(es_model)        # scalar fitness
        fit_scores.append(s)

        _restore_inplace(es_model, noises, SIGMA)

    # 2) Normalize fitness (z-score)
    fit_scores = np.asarray(fit_scores, dtype=np.float32)
    mean, std = fit_scores.mean(), fit_scores.std() + 1e-8
    z = (fit_scores - mean) / std

    # 3) Aggregate update
    _aggregate_update(es_model, seeds, z, SIGMA, LR)

    if t % EVAL_EVERY == 0:
        eval_score = es_objective(es_model)

        # Optional: mirror your preview printout (new names vs liked_names)
        preview = generate(es_model, device=device, seed=42 + t)
        new_names = set(preview) - set(liked_names)
        print(f"step = {t:2d} popS={mean:.4f}\u00b1{std:.4f} evalS={eval_score:.4f} net_new_names={len(new_names):2d}")
        print(",".join(list(new_names)[:10]))
        print()

after = es_objective(es_model)
print(f"After  ES objective (reward-weighted log-lik): {after:.4f}")